# IT5006- Getting started Lab on GX
## Great Expectations: Data Quality for Olist

**Duration:** ~30 minutes

---

### 🎯 Learning Objectives
1. **Connect GX to SQLite** — validate data directly from the course database
2. **Define expectations** for real e-commerce tables (orders, items, payments)
3. **Detect data quality issues** with **row-level identification**
4. **Implement pipeline decision patterns** — gate before KPI queries

---

### Industry Scenario
> *"Before running your weekly KPI dashboard queries, you need to validate that the source data meets quality standards. Bad data → wrong metrics → wrong decisions."*

### ⚠️ Why This Matters
- GX is your **gate** before data enters dashboards and ML models

### Key Expectations for Olist

| Table | Critical Checks | Why It Matters |
|-------|-----------------|----------------|
| `orders` | order_id unique, customer_id not null, valid status | Cohort analysis, joins |
| `order_items` | price/freight positive, order_id not null | GMV accuracy |
| `order_payments` | payment_value positive, valid type | Revenue reporting |

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Part A: Setup

In [ ]:
# Install Great Expectations (takes ~1 minute)
!pip install great_expectations -q
print("Great Expectations installed!")

In [ ]:
import sqlite3
import pandas as pd
import great_expectations as gx
import great_expectations.expectations as gxe
import inspect

# Connect to the ACTUAL Olist database (same as Lab 2.0)
DB_PATH = "Your-Path-to/Olist/olist.sqlite" # add your own path on the drive
conn = sqlite3.connect(DB_PATH)

print(f"Connected to {DB_PATH}")
print(f"GX version: {gx.__version__}")

In [ ]:
# Quick schema check (same tables from Lab 2.0)
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)
print("Olist Tables:")
for t in tables['name']:
    n = pd.read_sql(f"SELECT COUNT(*) as n FROM {t}", conn)['n'][0]
    print(f"   {t}: {n:,} rows")

## Part B: Connect GX to Olist Data
We'll validate the **orders** table — the core of all KPI queries from Lab 2.0.

### GX Component Hierarchy
```
Data Context          ← Entry point
    └── Data Source   ← Our SQLite connection  
        └── Data Asset   ← orders table
            └── Batch Definition   ← All rows
```

#### Note:
All GX workflows start with the creation of a Data Context. A Data Context is the Python object that serves as an entrypoint for the GX Core Python library, and it also manages the settings and metadata for your GX workflow.

- 'Ephemeral' means it's an in-memory context, perfect for quick, non-persistent data quality checks within a script or notebook without needing to save configuration files to disk.

In [ ]:
# Create GX context
context = gx.get_context(mode="ephemeral")

# print(f"contect info: {context}")

In [ ]:
# Load orders table from SQLite
orders_df = pd.read_sql("SELECT * FROM orders", conn)
print(f"Loaded orders table: {len(orders_df):,} rows, {len(orders_df.columns)} columns")
orders_df.head(3)

### Next:
Next, you create
- a Data Source: A Data Source represents the connection to your data environment,
- Data Asset: A Data Asset is a logical collection of records within a Data Source that you care about. It specifies what specific data you want to access., and
- Batch Definition: You then use the Batch Definition to generate a Batch of data to validate.

- In Great Expectations (GX), these three components form a hierarchy that defines how the framework finds, organizes, and retrieves your data for validation



### Note:
GX components are unique on name. Once a component is created with the Data Context, adding another component with the same name will cause an error. To enable repeated execution of cookbook cells that add GX workflow components, you shold use the following pattern as best practice:

    try:
        Add a new component(s) to the context
    except:
        Get component(s) from the context by name

In [ ]:
# Configure GX data source
try:
    pandas_source = context.data_sources.add_pandas(name="olist_source")
except Exception:
    pandas_source = context.get_datasource(name="olist_source")
# asset: A specific table in a database, a set of files in a folder matching a regex, or a specific Pandas DataFrame.
try:
    orders_asset = pandas_source.add_dataframe_asset(name="orders")
except Exception:
    orders_asset = pandas_source.get_asset(name="orders")
# A Batch Definition specifies how to organize and retrieve the data from a Data Asset into individual chunks called Batches.
orders_batch = orders_asset.add_batch_definition_whole_dataframe("orders_batch")

print("GX data source configured for orders table")

### Try: Defining a Batch for only 'delivered' orders

Instead of validating the entire `orders_df`, let's say you only want to check the quality of orders that have been 'delivered'. You can create a new batch definition that filters the `orders_df` accordingly.

## Part C: Define Expectations for Orders

An Expectation is a simple, declarative, verifiable assertion about your data.

You can validate a Batch of data using an Expectation.

Available Expectations can be easily found and instantiated using the gxe alias defined in the imports.

In [ ]:
# Get all classes in the gx.expectations module that are valid Expectations
expectations_list = [
    name for name, obj in inspect.getmembers(gx.expectations)
    if inspect.isclass(obj) and name.startswith("Expect")
]

# Print a sorted list
for exp in sorted(expectations_list):
    print(exp)

- First, create an Expectation that expects the columns in the customer data to match the provided ordered list of column names.

**Think about Lab 2.0 KPI queries:**
- GMV requires `order_status = 'delivered'` — what if status values are inconsistent?
- On-time delivery needs `order_delivered_customer_date` — what if it's NULL?
- Cohort analysis uses `order_purchase_timestamp` — what if dates are invalid?

**Our data contracts:**

In [ ]:
# Create Expectation Suite for Orders table
# It is a logical collection of Expectations that together describe the "ideal" or healthy state of a specific dataset
orders_suite = gx.ExpectationSuite(name="orders_quality_checks")

expectations = [
    # === RULE 1: Primary key must be unique ===
    gxe.ExpectColumnValuesToBeUnique(column="order_id"),

    # === RULE 2: Foreign key must not be NULL ===
    gxe.ExpectColumnValuesToNotBeNull(column="customer_id"),

    # === RULE 3: Purchase timestamp must not be NULL (needed for all time-based queries) ===
    gxe.ExpectColumnValuesToNotBeNull(column="order_purchase_timestamp"),

    # === RULE 4: Order status must be from known values ===
    # These are the statuses we see in Lab 2.0
    gxe.ExpectColumnValuesToBeInSet(
        column="order_status",
        value_set=["created", "approved", "invoiced", "processing", "shipped", "delivered", "canceled", "unavailable"]
    ),

    # === RULE 5: Row count sanity check ===
    gxe.ExpectTableRowCountToBeBetween(min_value=1000, max_value=500000)
]

for expectation in expectations:
    orders_suite.add_expectation(expectation)

# Remove suite if it already exists before adding
# We'll try to delete the suite and gracefully handle if it doesn't exist.
try:
    context.suites.delete(name=orders_suite.name)
    print(f"Removed existing suite: {orders_suite.name}")
except Exception as e:
    # This catches the error if the suite does not exist, allowing 'add' to proceed.
    pass

# Add suite to context
orders_suite = context.suites.add(orders_suite)

print("Orders Expectation Suite created!")
print("\nRules defined:")
for i, exp in enumerate(orders_suite.expectations, 1):
    # Correctly access column from the expectation object itself if it exists
    col = getattr(exp, 'column', 'table-level')
    print(f"   {i}. {type(exp).__name__} → {col}")

## Part D: Validate the Olist Orders Data

Let's run validation on the real Olist data and see what we find!

- Validation: gx.ValidationDefinition is the "glue" that binds Data and Logic together into an executable task.

- While a Data Asset defines where the data is, and an Expectation Suite defines what the data should look like, the Validation Definition defines the specific action: **"Run this specific suite against this specific batch of data."**

How it fits in the Workflow:

The Validation Definition is the final step before actually generating results.

The standard flow is:
1.   Batch Definition: Selects the data.
1.   Expectation Suite: Defines the rules.
1.   Validation Definition: Pairs them.
1.   Action: You call .run() on this definition to get a ValidationResult


In [ ]:
# Create validation definition
orders_validation = gx.ValidationDefinition(
    name="orders_validation",
    data=orders_batch,
    suite=orders_suite
)
orders_validation = context.validation_definitions.add(orders_validation)

# Run validation on Olist orders
results = orders_validation.run(batch_parameters={"dataframe": orders_df})
print("\n")
print(type(results))
# Show results
print("=" * 60)
if results.success:
    print("✅ ALL CHECKS PASSED!")
else:
    print("⚠️ SOME CHECKS FAILED - Issues detected in Olist data")
print("=" * 60)

print(f"\n Results Summary:")
print(f"Total expectations: {results.statistics.get('evaluated_expectations', 'N/A')}")
print(f"Passed: {results.statistics.get('successful_expectations', 'N/A')}")
print(f"Failed: {results.statistics.get('unsuccessful_expectations', 'N/A')}")

GX returns an `ExpectationSuiteValidationResult` object, as seen in the output. This object provides comprehensive metadata about the validation run for an entire suite of expectations.

It can be accessed like a dictionary and contains various fields. Most critically, the `success` field indicates whether all expectations within the suite passed. It also contains a list of individual `ExpectationValidationResult` objects, one for each expectation, allowing for detailed inspection of each check's outcome.

In [ ]:
# Detailed results for each expectation with ROW-LEVEL identification
print("Detailed Results:\n")

for result in results.results:
    exp_type = result.expectation_config.type
    success = result.success
    status = "✅ PASS" if success else "❌ FAIL"

    kwargs = result.expectation_config.kwargs
    col = kwargs.get('column', 'table-level')

    print(f"{status}: {exp_type}")
    print(f"       Column: {col}")

    if not success:
        res = result.result
        if 'unexpected_count' in res:
            print(f"       Unexpected count: {res['unexpected_count']} rows")
        if 'partial_unexpected_list' in res:
            bad_values = res['partial_unexpected_list'][:5]
            print(f"       Sample bad values: {bad_values}")

            # Show actual rows with issues
            if col != 'table-level' and bad_values:
                print(f"\n Sample rows with '{col}' issues:")
                bad_rows = orders_df[orders_df[col].isin(bad_values)].head(5)
                display_cols = ['order_id', col, 'order_status', 'order_purchase_timestamp']
                display_cols = [c for c in display_cols if c in orders_df.columns]
                print(bad_rows[display_cols].to_string(index=False))
    print()

**Note:** The Olist data is clean! All orders pass validation. Let's inspect some more tables...

## Part E: To-Do - Validate Order Items (GMV source)
The `order_items` table is critical for GMV calculations. Let's validate:
- `price` and `freight_value` must be positive
- `order_id` must exist (foreign key integrity)

In [ ]:
# Load order_items table

# Configure GX for order_items

# Create expectations for order_items


# Price must be positive (negative would break GMV)

# Freight must be non-negative


# order_id must not be null (foreign key)

# product_id must not be null

print("Order Items Expectation Suite created!")

In [ ]:
# Validate order_items


# Detailed results with ROW-LEVEL identification


## Part F: To-Do - Validate Order Payments

**Your turn!** The `order_payments` table is used for paid revenue calculations.

Create expectations for:
1. `payment_value` must be positive (0 to 50000)
2. `payment_type` must be in: credit_card, boleto, voucher, debit_card
3. `order_id` must not be NULL

### How
- How GX catches unexpected values in production
- How to identify the exact rows and values causing failures


In [ ]:
# TODO: Load payments data
payments_df = pd.read_sql("SELECT * FROM order_payments", conn)
print(f"Loaded order_payments: {len(payments_df):,} rows")

# TODO: Create expectations suite for payments
# Uncomment and complete:

# payments_suite = gx.ExpectationSuite(name="payments_quality")
#
# payments_suite.add_expectation(gxe.ExpectColumnValuesToBeBetween(
#     column="???",
#     min_value=???,
#     max_value=???
# ))
#
# payments_suite.add_expectation(gxe.ExpectColumnValuesToBeInSet(
#     column="???",
#     value_set=[???]
# ))
#
# payments_suite.add_expectation(gxe.ExpectColumnValuesToNotBeNull(column="???"))

---

## Summary

1. **Connected GX to real Olist SQLite** — same database as Lab 2.0
2. **Validated 3 tables**: orders, order_items, order_payments
3. **Defined business-relevant expectations** (nulls, ranges, valid values)




In [ ]:
# Cleanup
conn.close()
print("Database connection closed")

---

## Resources

- [Great Expectations Docs](https://docs.greatexpectations.io/)
- [Expectation Gallery](https://greatexpectations.io/expectations/) — 300+ built-in checks
- [GX Pipeline Tutorial](https://github.com/greatexpectationslabs/tutorial-gx-in-the-data-pipeline) — Airflow integration

---



## END Additional Lab 2